# Sleeper league model: icemen (krishd20)
Pulls from the public Sleeper API snapshot in `data/`, trains a next-week points model,
simulates the rest of the regular season, and writes `dashboard_data.json` for the dashboard.

In [1]:
import json, math, random
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

D = Path('/home/claude/ff/data')
J = lambda f: json.load(open(D / f))
league = J('league.json'); rosters = J('rosters.json')
users = {u['user_id']: u for u in J('users.json')}
P = J('players.json')
state = J('state.json')
CUR = int(state['week'])            # week in progress
LAST_REG = league['settings']['playoff_week_start'] - 1   # 14
PO_WEEKS = [15, 16, 17]
ME = '1270089224720232448'
POS = ['QB', 'RB', 'WR', 'TE', 'K', 'DEF']
stats = {w: J(f'stats{w}.json') for w in range(1, CUR + 1)}
proj = {w: {r['player_id']: r for r in J(f'proj{w}.json')} for w in range(1, 19)}
mups = {w: J(f'm{w}.json') for w in range(1, LAST_REG + 1)}
trend = {t['player_id']: t['count'] for t in J('trend_add.json')}

def pinfo(pid):
    p = P.get(pid, {})
    if pid.isalpha():  # team defense
        return dict(name=f'{pid} D/ST', pos='DEF', team=pid, inj=None, body=None, depth=None, status='Active')
    return dict(name=p.get('full_name') or pid, pos=p.get('position'), team=p.get('team'),
                inj=p.get('injury_status'), body=p.get('injury_body_part'), depth=p.get('depth_chart_order'), status=p.get('status'))

def team_name(r):
    u = users.get(r['owner_id'], {})
    return (u.get('metadata') or {}).get('team_name') or f"Team {u.get('display_name')}"

teams = {}
for r in rosters:
    s = r['settings']
    teams[r['roster_id']] = dict(rid=r['roster_id'], name=team_name(r), user=users[r['owner_id']]['display_name'],
        div=s.get('division'), w=s['wins'], l=s['losses'], t=s.get('ties', 0),
        pf=s['fpts'] + s.get('fpts_decimal', 0) / 100, pa=s.get('fpts_against', 0) + s.get('fpts_against_decimal', 0) / 100,
        faab_left=league['settings']['waiver_budget'] - s.get('waiver_budget_used', 0),
        players=[p for p in (r['players'] or [])], reserve=r.get('reserve') or [], starters=r['starters'],
        me=r['owner_id'] == ME)
MY = next(t['rid'] for t in teams.values() if t['me'])
DIVN = {1: league['metadata']['division_1'], 2: league['metadata']['division_2'], 3: league['metadata']['division_3']}
print(teams[MY]['name'], teams[MY]['w'], teams[MY]['l'], round(teams[MY]['pf'], 2))

icemen 2 3 593.58


## Player-week table
One row per player per week played. Features use only information available before that week.

In [2]:
def pts(w, pid):
    s = stats[w].get(pid)
    if not s or not s.get('gms_active'): return None
    return s.get('pts_ppr', 0.0)

def opp_of(w, pid):
    r = proj[w].get(pid)
    return r.get('opponent') if r else None

# fantasy points allowed by each defense to each position (through week w-1)
allowed = []
for w in range(1, CUR):
    for pid, s in stats[w].items():
        pos = pinfo(pid)['pos']
        if pos not in POS or not s.get('gms_active'): continue
        o = opp_of(w, pid)
        if o: allowed.append((w, o, pos, s.get('pts_ppr', 0.0)))
allowed = pd.DataFrame(allowed, columns=['week', 'def', 'pos', 'pts'])
by_def_week = allowed.groupby(['week', 'def', 'pos']).pts.sum().reset_index()

def dvp(def_team, pos, upto):
    d = by_def_week[(by_def_week.week < upto) & (by_def_week.pos == pos)]
    if d.empty or def_team is None: return np.nan
    lg = d.groupby('def').pts.mean().mean()
    mine = d[d['def'] == def_team].pts.mean()
    return mine / lg if lg and not np.isnan(mine) else np.nan

def history(pid, upto):
    """Form features from weeks < upto."""
    g = [w for w in range(1, upto) if pts(w, pid) is not None]
    if not g:
        return dict(gp=0, avg=np.nan, last=np.nan, opp=np.nan, snap=np.nan, ratio=np.nan, sd=np.nan)
    p = [pts(w, pid) for w in g]
    opps, snaps, ratios = [], [], []
    for w in g:
        s = stats[w][pid]
        opps.append(s.get('rec_tgt', 0) + s.get('rush_att', 0) + 0.25 * s.get('pass_att', 0))
        if s.get('tm_off_snp'): snaps.append(s.get('off_snp', 0) / s['tm_off_snp'])
        pr = proj[w].get(pid, {}).get('stats', {}).get('pts_ppr')
        if pr and pr > 1: ratios.append(pts(w, pid) / pr)
    return dict(gp=len(g), avg=np.mean(p), last=p[-1], opp=np.mean(opps),
                snap=np.mean(snaps) if snaps else np.nan,
                ratio=np.mean(ratios) if ratios else np.nan, sd=np.std(p) if len(p) > 1 else np.nan)

POSI = {p: i for i, p in enumerate(POS)}
def features(pid, w, upto):
    """Features to predict week w using data from weeks < upto."""
    pi = pinfo(pid)
    pr = proj[w].get(pid, {}).get('stats', {}).get('pts_ppr')
    h = history(pid, upto)
    return dict(proj=pr if pr is not None else np.nan, pos=POSI.get(pi['pos'], -1),
                dvp=dvp(opp_of(w, pid), pi['pos'], upto), **h)

rows = []
for w in range(1, CUR):
    for pid in stats[w]:
        if pinfo(pid)['pos'] not in POS: continue
        y = pts(w, pid)
        if y is None or pid not in proj[w]: continue
        f = features(pid, w, w); f.update(y=y, week=w, pid=pid); rows.append(f)
df = pd.DataFrame(rows)
FEATS = ['proj', 'pos', 'dvp', 'gp', 'avg', 'last', 'opp', 'snap', 'ratio', 'sd']
print(df.shape, df.groupby('week').size().to_dict())

(3625, 13) {1: 721, 2: 734, 3: 735, 4: 737, 5: 698}


## Model bake-off on held-out weeks
Each candidate trains on weeks before the test week and predicts it. Candidates: Sleeper (Rotowire) projection,
gradient-boosted trees on form + matchup features, a GBM that corrects the projection's residual, and a
Bayesian shrink of the projection toward the player's season average. Lowest mean MAE across weeks 4 and 5 wins.

In [3]:
def gbm(**k):
    kw = dict(max_iter=200, learning_rate=0.03, max_leaf_nodes=8, min_samples_leaf=60, l2_regularization=3, random_state=0); kw.update(k)
    return HistGradientBoostingRegressor(**kw)

def candidates(tr, X):
    out = {'projection': X.proj.values}
    out['gbm'] = gbm().fit(tr[FEATS], tr.y).predict(X[FEATS])
    out['residual_gbm'] = X.proj.values + 0.3 * gbm().fit(tr[FEATS], tr.y - tr.proj).predict(X[FEATS])
    w = X.gp / (X.gp + 10); out['shrink_to_avg'] = ((1 - w) * X.proj + w * X.avg.fillna(X.proj)).values
    return out

base = df[df.proj.notna() & (df.proj > 0.5) & (df.week >= 2)]
bake = {}
for tw in [CUR - 2, CUR - 1]:
    tr, te = base[base.week < tw], base[base.week == tw]
    for k, v in candidates(tr, te).items():
        bake.setdefault(k, []).append(float(np.mean(np.abs(te.y - v))))
mae = {k: round(float(np.mean(v)), 3) for k, v in bake.items()}
best = min(mae, key=mae.get)
print(mae, '-> using', best)

{'projection': 4.39, 'gbm': 4.677, 'residual_gbm': 4.432, 'shrink_to_avg': 4.468} -> using projection


## Calibrated floor and ceiling
Ratio of actual to projected points, by position, gives empirical 20th/80th percentile multipliers.
Checked out of sample: fit on weeks 1 to 4, measure how often week 5 lands inside the band (target 60%).

In [4]:
def ratio_q(d):
    d = d[d.proj > 3]
    return {pos: (float(np.quantile(g.y / g.proj, 0.2)), float(np.quantile(g.y / g.proj, 0.8))) for pos, g in d.groupby('pos')}
q_check = ratio_q(base[base.week < CUR - 1]); te = base[(base.week == CUR - 1) & (base.proj > 3)]
lo_c = te.proj * te.pos.map(lambda p: q_check[p][0]); hi_c = te.proj * te.pos.map(lambda p: q_check[p][1])
coverage = float(np.mean((te.y >= lo_c) & (te.y <= hi_c)))
RQ = ratio_q(base)
print('band coverage on week 5:', round(coverage, 3), {list(POSI)[k]: tuple(round(x, 2) for x in v) for k, v in RQ.items()})
te_pred = candidates(base[base.week < CUR - 1], te)[best]
resid_sd = (te.assign(r=te.y - te_pred).groupby('pos').r.std().rename(index={v: k for k, v in POSI.items()}).to_dict())
print({k: round(v, 1) for k, v in resid_sd.items()})

band coverage on week 5: 0.577 {'QB': (0.69, 1.46), 'RB': (0.39, 1.42), 'WR': (0.38, 1.64), 'TE': (0.27, 1.58), 'K': (0.59, 1.69)}


{'QB': 7.9, 'RB': 6.2, 'WR': 6.6, 'TE': 6.4, 'K': 5.5}


In [5]:
def predict_rows(pids, weeks):
    rows = []
    for pid in pids:
        for w in weeks:
            f = features(pid, w, CUR); f.update(pid=pid, week=w); rows.append(f)
    X = pd.DataFrame(rows)
    has = X.proj.notna() & (X.proj > 0)
    Xf = X.assign(proj=X.proj.fillna(0))
    mu = candidates(base, Xf)[best]
    X['mu'] = np.where(has, np.maximum(mu, 0), 0.0)
    lo_q = X.pos.map(lambda p: RQ.get(p, (0.5, 1.5))[0]); hi_q = X.pos.map(lambda p: RQ.get(p, (0.5, 1.5))[1])
    X['lo'] = X.mu * lo_q; X['hi'] = X.mu * hi_q
    return X

## Weekly expected points for every relevant player (weeks 6 to 17)
Injury handling: Out / IR / PUP / Suspended score 0 this week; Doubtful x0.3; Questionable x0.85.
Future weeks rely on the projection feed, which zeros out players expected to miss time.

In [6]:
rostered = {p for t in teams.values() for p in t['players']}
fa_pool = [pid for pid, p in P.items() if p.get('team') and p.get('position') in POS and p.get('active')
           and pid not in rostered and pid in proj[CUR + 1]]
fa_pool += [d for d in {r['player_id'] for r in proj[CUR + 1].values() if r['player']['position'] == 'DEF'} if d not in rostered]
fa_pool = sorted(set(fa_pool))
WEEKS = list(range(CUR, 18))
pred_df = predict_rows(sorted(rostered | set(fa_pool)), WEEKS)

played6 = {pid for pid, s in stats[CUR].items() if s.get('gms_active')}
INJ = {'Out': 0, 'IR': 0, 'PUP': 0, 'Sus': 0, 'Suspended': 0, 'Doubtful': 0.3, 'Questionable': 0.85}
def adj(row):
    pid, w = row.pid, row.week
    if w == CUR and pid in played6:   # game already played this week: use actual
        a = stats[CUR][pid].get('pts_ppr', 0.0); return pd.Series([a, a, a])
    inj = pinfo(pid)['inj']
    k = INJ.get(inj, 1.0) if w == CUR else (0 if inj == 'IR' and w < CUR + 4 else 1.0)
    return pd.Series([row.mu * k, row.lo * k, row.hi * k])
pred_df[['mu', 'lo', 'hi']] = pred_df.apply(adj, axis=1)
EXP = {(r.pid, r.week): (r.mu, r.lo, r.hi) for r in pred_df.itertuples()}
print(len(fa_pool), 'free agents scored')

702 free agents scored


## Lineup optimizer
QB, 2 RB, 2 WR, TE, 2 FLEX (RB/WR/TE), K, DEF. Greedy fill of fixed slots then best two remaining
flex-eligible is optimal for this slot structure.

In [7]:
SLOTS = dict(QB=1, RB=2, WR=2, TE=1, K=1, DEF=1); FLEX = 2
def best_lineup(pids, w, key=0):
    by = {}
    for p in pids:
        by.setdefault(pinfo(p)['pos'], []).append((EXP.get((p, w), (0, 0, 0))[key], p))
    for v in by.values(): v.sort(reverse=True)
    chosen, total, rest = [], 0.0, []
    for pos, n in SLOTS.items():
        lst = by.get(pos, [])
        for v, p in lst[:n]: chosen.append((pos, p, v)); total += v
        if pos in ('RB', 'WR', 'TE'): rest += lst[n:]
    rest.sort(reverse=True)
    for v, p in rest[:FLEX]: chosen.append(('FLEX', p, v)); total += v
    return total, chosen

def active(t): return [p for p in t['players'] if p not in t['reserve']]
def ros_value(pids, weeks=range(CUR + 1, LAST_REG + 1)):
    return sum(best_lineup(pids, w)[0] for w in weeks)

for t in teams.values():
    t['exp_week'] = {w: best_lineup(active(t) + t['reserve'] if w > CUR + 3 else active(t), w)[0] for w in WEEKS}
print({teams[r]['name']: round(teams[r]['exp_week'][CUR + 1], 1) for r in teams})

{'icemen': 131.9, 'Sussybakamogus': 138.0, 'Yalam': 122.6, 'Tee liggin': 117.0, 'Gonna be a long season': 139.2, 'Team RayaanP': 118.2, 'Team Nishzzz': 127.1, 'Team arnavn23': 141.3, 'killa k im gay': 142.7, 'Saul': 117.0, 'Team aydmaz15': 125.6, 'Team Arjunraj': 120.6}


## Season simulation (20,000 runs)
Week 6 uses each team's set starters (actual points where the game is done). Later weeks assume an optimal lineup.
Team score sd comes from the week-5 holdout residuals summed across a lineup.

In [8]:
team_sd = math.sqrt(sum(resid_sd.get(pos, 7) ** 2 * n for pos, n in {**SLOTS, 'FLEX': 0}.items())
                    + FLEX * resid_sd.get('WR', 7) ** 2)
print('team weekly sd', round(team_sd, 1))

def wk6_mean(t):
    m = next(x for x in mups[CUR] if x['roster_id'] == t['rid'])
    return sum(EXP.get((p, CUR), (0,))[0] for p in m['starters'] if p != '0')
for t in teams.values(): t['exp_week'][CUR] = wk6_mean(t)

sched = {w: {} for w in range(CUR, LAST_REG + 1)}
for w in range(CUR, LAST_REG + 1):
    g = {}
    for x in mups[w]: g.setdefault(x['matchup_id'], []).append(x['roster_id'])
    sched[w] = [tuple(v) for v in g.values() if len(v) == 2]

def seed(rec):
    """rec: rid -> (wins, pf). Returns ordered list of 8 playoff rids."""
    key = lambda r: (rec[r][0], rec[r][1])
    winners = [max([r for r in teams if teams[r]['div'] == d], key=key) for d in DIVN]
    winners.sort(key=key, reverse=True)
    wild = sorted([r for r in teams if r not in winners], key=key, reverse=True)[:5]
    return winners + wild

def simulate(team_means, n=20000, rng=np.random.default_rng(7)):
    rids = sorted(teams)
    idx = {r: i for i, r in enumerate(rids)}
    weeks = list(range(CUR, LAST_REG + 1))
    mu = np.array([[team_means[r][w] for w in weeks] for r in rids])          # teams x weeks
    played_frac = {r: 0 for r in rids}
    sd = np.array([[team_sd * (0.85 if w == CUR else 1.0) for w in weeks] for r in rids])
    scores = rng.normal(mu[None], sd[None], size=(n, len(rids), len(weeks)))
    wins = np.tile(np.array([teams[r]['w'] + 0.5 * teams[r]['t'] for r in rids], float), (n, 1))
    pf = np.tile(np.array([teams[r]['pf'] for r in rids]), (n, 1)) + scores.sum(2)
    for j, w in enumerate(weeks):
        for a, b in sched[w]:
            ia, ib = idx[a], idx[b]
            aw = scores[:, ia, j] > scores[:, ib, j]
            wins[:, ia] += aw; wins[:, ib] += ~aw
    made = np.zeros(len(rids)); divw = np.zeros(len(rids)); seeds = np.zeros((len(rids), 8))
    # vectorized seeding: division winners then wildcards, tiebreak PF
    key = wins * 10000 + pf
    divs = np.array([teams[r]['div'] for r in rids])
    win_mask = np.zeros_like(key, bool)
    for d in DIVN:
        cols = np.where(divs == d)[0]
        best_c = cols[np.argmax(key[:, cols], 1)]
        win_mask[np.arange(n), best_c] = True
    k_div = np.where(win_mask, key, -1e12); k_wild = np.where(win_mask, -1e12, key)
    order_div = np.argsort(-k_div, 1)[:, :3]; order_wild = np.argsort(-k_wild, 1)[:, :5]
    order = np.concatenate([order_div, order_wild], 1)
    for s in range(8):
        np.add.at(seeds, (order[:, s], s), 1)
    made = seeds.sum(1) / n; divw = win_mask.mean(0)
    return {r: dict(playoff=float(made[idx[r]]), divwin=float(divw[idx[r]]), exp_w=float(wins[:, idx[r]].mean()),
                    seeds=(seeds[idx[r]] / n).round(4).tolist(), w10=float(np.percentile(wins[:, idx[r]], 10)),
                    w90=float(np.percentile(wins[:, idx[r]], 90))) for r in rids}

base_means = {r: teams[r]['exp_week'] for r in teams}
odds = simulate(base_means)
pd.DataFrame({teams[r]['name']: v for r, v in odds.items()}).T[['playoff', 'divwin', 'exp_w']].sort_values('playoff', ascending=False).round(3)

team weekly sd 20.9


,playoff,divwin,exp_w
killa k im gay,0.99945,0.9268,10.50755
Gonna be a long season,0.9989,0.94845,10.18145
Team arnavn23,0.988,0.82025,8.721
Yalam,0.90635,0.0486,7.73145
Team aydmaz15,0.81385,0.1516,7.00515
Team Nishzzz,0.72685,0.031,6.6945
Tee liggin,0.7129,0.0229,6.96795
icemen,0.6376,0.01785,6.2784
Saul,0.38575,0.02385,5.42305
Team Arjunraj,0.36095,0.0027,5.1784


## Clinch / elimination (math, not simulation)
Clinched: even losing out, fewer than 7 other teams can reach your win total.
Eliminated: at least 8 teams are already guaranteed more wins than your max and you cannot win your division.

In [9]:
left = {r: sum(1 for w in sched for g in sched[w] if r in g) for r in teams}
for r, t in teams.items():
    floor_w, ceil_w = t['w'], t['w'] + left[r]
    can_reach = sum(1 for o in teams if o != r and teams[o]['w'] + left[o] >= floor_w)
    t['clinched'] = can_reach < 7
    above = sum(1 for o in teams if o != r and teams[o]['w'] > ceil_w)
    div_rivals_above = any(teams[o]['w'] > ceil_w for o in teams if teams[o]['div'] == t['div'] and o != r)
    t['eliminated'] = above >= 8 and div_rivals_above
print('clinched', [t['name'] for t in teams.values() if t['clinched']], 'eliminated', [t['name'] for t in teams.values() if t['eliminated']])

clinched [] eliminated []


## Waiver targets for icemen
Gain = rise in your optimal-lineup points over weeks 6 to 14 after adding the player and dropping your least
valuable bench player. FAAB bid is a heuristic on that gain plus how hot the player is on Sleeper adds.

In [10]:
me = teams[MY]
mine = active(me)
reg = range(CUR, LAST_REG + 1)
def with_ir(pids, res, w): return pids + res if w >= CUR + 4 else pids
def season_total(pids): return sum(best_lineup(with_ir(pids, me['reserve'], w), w)[0] for w in reg)
base_total = season_total(mine)
def drop_cost(p): return base_total - season_total([x for x in mine if x != p])
drop_order = sorted(mine, key=drop_cost)
cands = sorted(fa_pool, key=lambda p: -sum(EXP.get((p, w), (0,))[0] for w in reg))[:150]
wv = []
for p in cands:
    gains = [(season_total([x for x in mine if x != d] + [p]) - base_total, d) for d in drop_order[:4]]
    g, d = max(gains)
    if g > 0.5: wv.append((g, p, d))
wv.sort(reverse=True)
faab = me['faab_left']
def starts(pids, p): return sum(1 for w in reg if any(x[1] == p for x in best_lineup(with_ir(pids, me['reserve'], w), w)[1]))
CAP = dict(QB=2, K=1, DEF=2, TE=3, RB=6, WR=6); used = {}
waivers = []
for g, p, d in wv:
    pos = pinfo(p)['pos']
    if used.get(pos, 0) >= CAP[pos] or len(waivers) >= 16: continue
    used[pos] = used.get(pos, 0) + 1
    pi = pinfo(p); nxt = proj[CUR].get(p) or proj[CUR + 1].get(p) or {}
    t = trend.get(p, 0)
    bid = int(round(faab * min(0.25, g / 250 + (0.03 if t > 500000 else 0))))
    ups = EXP.get((p, CUR + 1), (0, 0, 0))
    waivers.append(dict(pid=p, name=pi['name'], pos=pi['pos'], team=pi['team'], inj=pi['inj'], depth=pi['depth'],
        gain=round(g, 1), starts=starts([x for x in mine if x != d] + [p], p), drop=pinfo(d)['name'], drop_pid=d, bid=bid, trend=t,
        next_mu=round(ups[0], 1), next_lo=round(ups[1], 1), next_hi=round(ups[2], 1),
        opp=opp_of(CUR + 1, p), dvp=None if np.isnan(dvp(opp_of(CUR + 1, p), pi['pos'], CUR)) else round(dvp(opp_of(CUR + 1, p), pi['pos'], CUR), 2),
        played_this_week=p in played6))
pd.DataFrame(waivers)[['name', 'pos', 'team', 'gain', 'drop', 'bid', 'next_mu', 'next_hi', 'trend']].head(12)

,name,pos,team,gain,drop,bid,next_mu,next_hi,trend
0,Kyler Murray,QB,MIN,18.8,Germie Bernard,75,18.0,26.2,0
1,Cam Ward,QB,TEN,15.6,Germie Bernard,62,13.9,20.2,0
2,Will Reichard,K,MIN,8.2,Germie Bernard,33,9.4,15.9,0
3,CLE D/ST,DEF,CLE,1.0,Germie Bernard,4,8.6,12.9,0
4,NYJ D/ST,DEF,NYJ,0.6,Germie Bernard,3,8.3,12.4,0


## Win-win 1-for-1 trade ideas
Both lineups must gain over the rest of the regular season. Ranked by the smaller of the two gains,
since that is what makes the other manager say yes.

In [11]:
def tot(pids, res): return sum(best_lineup(with_ir(pids, res, w), w)[0] for w in range(CUR + 1, LAST_REG + 1))
my_base = tot(mine, me['reserve'])
trades = []
for r, t in teams.items():
    if r == MY: continue
    th = active(t); tb = tot(th, t['reserve'])
    for a in mine:
        for b in th:
            if pinfo(a)['pos'] == 'DEF' or pinfo(b)['pos'] in ('K', 'DEF'): continue
            dm = tot([x for x in mine if x != a] + [b], me['reserve']) - my_base
            if dm <= 2: continue
            do = tot([x for x in th if x != b] + [a], t['reserve']) - tb
            if do <= 0: continue
            new_mine = [x for x in mine if x != a] + [b]
            st = sum(1 for w in range(CUR + 1, LAST_REG + 1) if any(x[1] == b for x in best_lineup(with_ir(new_mine, me['reserve'], w), w)[1]))
            if st >= 4: trades.append(dict(give=a, get=b, team=r, dm=round(dm, 1), do=round(do, 1), starts=st))
trades.sort(key=lambda x: -min(x['dm'], x['do']))
seen, trade_ideas = set(), []
for x in trades:
    if x['get'] in seen: continue
    seen.add(x['get']); trade_ideas.append(x)
    if len(trade_ideas) == 8: break
[(pinfo(x['give'])['name'], '->', pinfo(x['get'])['name'], teams[x['team']]['name'], x['dm'], x['do']) for x in trade_ideas]

[("Wan'Dale Robinson", '->', 'Jared Goff', 'Team aydmaz15', 10.9, 7.8),
 ('Mark Andrews', '->', 'Brenton Strange', 'Saul', 4.6, 6.9),
 ('Hunter Henry', '->', 'Dalton Kincaid', 'Saul', 4.3, 8.5),
 ('TreVeyon Henderson', '->', 'Jalen Hurts', 'Tee liggin', 8.5, 3.0),
 ('Hunter Henry', '->', 'RJ Harvey', 'Saul', 2.3, 4.5),
 ('Ladd McConkey', '->', 'Patrick Mahomes', 'Tee liggin', 5.2, 2.2),
 ('Christian McCaffrey',
  '->',
  'Kenneth Walker',
  'Gonna be a long season',
  3.0,
  1.7),
 ('Ladd McConkey', '->', 'Josh Allen', 'Team aydmaz15', 34.5, 0.2)]

## Export for the dashboard

In [12]:
import datetime as _dt
_dvp_cache = {}
def dvp_c(def_team, pos):
    k = (def_team, pos)
    if k not in _dvp_cache:
        v = dvp(def_team, pos, CUR); _dvp_cache[k] = None if v is None or np.isnan(v) else round(float(v), 2)
    return _dvp_cache[k]
def pexport(pid):
    pi = pinfo(pid)
    h = history(pid, CUR)
    return dict(n=pi['name'], pos=pi['pos'], tm=pi['team'], inj=pi['inj'], body=pi['body'], dep=pi['depth'],
                mu=[round(EXP.get((pid, w), (0, 0, 0))[0], 2) for w in WEEKS],
                lo=[round(EXP.get((pid, w), (0, 0, 0))[1], 2) for w in WEEKS],
                hi=[round(EXP.get((pid, w), (0, 0, 0))[2], 2) for w in WEEKS],
                opp=[opp_of(w, pid) for w in WEEKS],
                dvp=[dvp_c(opp_of(w, pid), pi['pos']) for w in WEEKS],
                pts=[pts(w, pid) for w in range(1, CUR + 1)],
                played=pid in played6,
                avg=None if np.isnan(h['avg']) else round(h['avg'], 1),
                snap=None if np.isnan(h['snap']) else round(h['snap'], 3),
                tr=trend.get(pid, 0))
# free-agent pool for in-browser waiver math: best rest-of-season expected points, capped per position
ros = lambda p: sum(EXP.get((p, w), (0,))[0] for w in range(CUR, 18))
POOL_CAP = dict(QB=14, RB=40, WR=50, TE=22, K=10, DEF=12)
pool, cnt = [], {}
for p in sorted(fa_pool, key=lambda p: -ros(p)):
    pos = pinfo(p)['pos']
    if cnt.get(pos, 0) < POOL_CAP.get(pos, 0) and ros(p) > 0:
        pool.append(p); cnt[pos] = cnt.get(pos, 0) + 1
plist = sorted(rostered | {w['pid'] for w in waivers} | set(pool))
cur_mup = {x['roster_id']: x for x in mups[CUR]}
hist = []
for w in range(1, CUR + 1):
    by = {}
    for x in mups[w]: by.setdefault(x['matchup_id'], []).append(x)
    for g in by.values():
        if len(g) != 2: continue
        for a, b in (g, g[::-1]):
            hist.append(dict(week=w, rid=a['roster_id'], opp=b['roster_id'], pts=a['points'], opp_pts=b['points'], final=w < CUR))
out = dict(
    asof=_dt.datetime.now(_dt.timezone(_dt.timedelta(hours=-5))).strftime('%a %b %-d, %-I:%M %p CT'),
    week=CUR, weeks=WEEKS, last_reg=LAST_REG, me=MY, mae=mae, method=best, coverage=round(coverage, 3),
    team_sd=round(team_sd, 1), divisions=DIVN, slots=SLOTS, flex=FLEX,
    teams={r: dict(name=t['name'], user=t['user'], div=t['div'], w=t['w'], l=t['l'], pf=round(t['pf'], 2),
                   pa=round(t['pa'], 2), faab=t['faab_left'], players=active(t), reserve=t['reserve'],
                   starters=cur_mup[r]['starters'], live=cur_mup[r]['points'],
                   exp={w: round(v, 1) for w, v in t['exp_week'].items()},
                   clinched=t['clinched'], eliminated=t['eliminated'], **odds[r]) for r, t in teams.items()},
    sched={w: sched[w] for w in sched},
    players={p: pexport(p) for p in plist},
    waivers=waivers, trades=trade_ideas, history=hist,
)
json.dump(out, open('/home/claude/ff/dashboard_data.json', 'w'), default=lambda o: o.item() if hasattr(o, 'item') else str(o))
print('players exported', len(plist))

players exported 344


## Tests

In [13]:
s = json.load(open('/home/claude/ff/dashboard_data.json'))
for r, t in s['teams'].items():
    raw = next(x for x in rosters if str(x['roster_id']) == str(r))['settings']
    assert t['w'] == raw['wins'] and t['l'] == raw['losses']
tot_po = sum(t['playoff'] for t in s['teams'].values())
assert abs(tot_po - 8) < 1e-6, tot_po
assert abs(sum(t['divwin'] for t in s['teams'].values()) - 3) < 1e-6
_, lu = best_lineup(mine, CUR + 1)
assert len(lu) == 10 and sum(1 for x in lu if x[0] == 'FLEX') == 2
assert all(pinfo(x[1])['pos'] in ('RB', 'WR', 'TE') for x in lu if x[0] == 'FLEX')
assert all(w['gain'] > 0 for w in waivers)
print('all tests pass; playoff probs sum to', round(tot_po, 3))

all tests pass; playoff probs sum to 8.0
